# Cluster quality comparison: paragraph baseline vs. graph-aware partition

This notebook evaluates the fetched clusters from **1.0_context_analysis.ipynb** against the final clusters from **1.7_graph_construction_node_attribute_fixed.ipynb**. The two methods operate on related but different units: the baseline clusters 2,534 reasoning paragraphs, while the graph-aware method partitions 3,941 extracted argument components. Comparisons therefore emphasize scale-independent quality measures and explicitly report coverage.

The later notebook remains unchanged. Its rerun outputs are read from `cluster_comparison_artifacts/`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from scipy import sparse
from sklearn.cluster import DBSCAN, HDBSCAN
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_distances
from transformers import AutoModel, AutoTokenizer

RANDOM_SEED = 7
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DOCUMENT_IDS = [
    "116062204", "116075685", "116080900", "116084352", "116084886", "116099338",
    "116102314", "116105977", "116106432", "116109598", "116110555", "116116081",
    "116118921", "116120654", "116121714", "116125191", "116126076", "116127831",
    "116141779", "116142769", "116149190", "116150283", "116167789", "116168634",
    "116170057", "116170822", "116192026", "116194637", "116196887", "116197578",
    "116199866", "116204676", "116213690", "116214232", "116220471", "116221948",
    "116226431", "116229939", "116232414", "116233794", "116245112", "116246991",
    "116248468", "116252607", "116253514", "116256069", "116257159", "116259036",
    "116260246", "116261015", "116261263", "116270112", "116277632", "116278949",
    "116283678", "116285648", "116289180", "116308208", "116310154", "116320626",
    "116321603", "116336690", "116339406", "116344031", "116350541", "116350825",
    "116358099", "116359089", "116376162", "116377931", "116385451", "116398308",
    "116410200", "116410462", "116414218", "116432495", "116434259", "116442566",
    "116462199", "116588653", "116681882", "116764649", "116780853", "116790074",
    "116794466", "116972204", "117069850", "117125134", "117130242", "117230819",
    "117251852", "117420480", "117440182", "117456818", "117460145", "117464049",
    "117489745", "117516936", "117556435",
]


def locate(relative: Path) -> Path:
    for root in (Path.cwd(), *Path.cwd().parents):
        for candidate in (
            root / relative,
            root / "paragraph_analysis" / relative,
            root / "src" / "legal_data_doc_parts" / relative,
            root / "src" / "legal_data_doc_parts" / "paragraph_analysis" / relative,
        ):
            if candidate.exists():
                return candidate.resolve()
    raise FileNotFoundError(relative)


ARTIFACT_DIR = locate(Path("cluster_comparison_artifacts"))
PARAGRAPH_DIR = locate(Path("doc_artifacts"))
ENCODER_DIR = locate(Path("bge_model/bge-small-uk-law-v1_1/final/encoder"))
TOKENIZER_DIR = locate(Path("bge_model/bge-small-uk-law-v1_1/final/tokenizer"))
print(f"Comparison artifacts: {ARTIFACT_DIR}")

Comparison artifacts: /home/vegura/Projects/study/legal-data-doc-parts/src/legal_data_doc_parts/paragraph_analysis/cluster_comparison_artifacts


/home/vegura/.cache/pypoetry/virtualenvs/legal-data-doc-parts-2qBMspuS-py3.14/lib64/python3.14/site-packages/torch/cuda/__init__.py:188: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 804: forward compatibility was attempted on non supported HW (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


## Reproduce the original paragraph baseline

The baseline retains the first notebook's core idea: mean-pooled normalized legal-language embeddings, cosine distance, and density clustering. The original fixed DBSCAN configuration is evaluated as saved. A small declared grid is also evaluated because the saved configuration was chosen from a plot and assigned most observations to noise.

In [2]:
paragraph_frames = [pd.read_parquet(PARAGRAPH_DIR / f"{doc_id}.parquet") for doc_id in DOCUMENT_IDS]
paragraphs = pd.concat(paragraph_frames, ignore_index=True)
reasoning = paragraphs.loc[paragraphs["section"].eq("reasoning")].copy().reset_index(drop=True)

tokenizer = AutoTokenizer.from_pretrained(str(TOKENIZER_DIR), local_files_only=True)
encoder = AutoModel.from_pretrained(
    str(ENCODER_DIR), local_files_only=True, add_pooling_layer=False,
).to(DEVICE).eval()

@torch.inference_mode()
def encode_texts(texts: list[str], batch_size: int = 64) -> np.ndarray:
    batches = []
    for start in range(0, len(texts), batch_size):
        tokens = tokenizer(
            texts[start:start + batch_size], padding=True, truncation=True, return_tensors="pt"
        )
        tokens = {name: value.to(DEVICE) for name, value in tokens.items()}
        token_embeddings = encoder(**tokens).last_hidden_state
        mask = tokens["attention_mask"].unsqueeze(-1).to(token_embeddings.dtype)
        pooled = (token_embeddings * mask).sum(1) / mask.sum(1).clamp_min(1.0)
        batches.append(F.normalize(pooled, p=2, dim=1).cpu())
    return torch.cat(batches).numpy()

baseline_embeddings = encode_texts(reasoning["text"].tolist())
baseline_distances = cosine_distances(baseline_embeddings).astype(np.float32)
np.fill_diagonal(baseline_distances, 0.0)
baseline_distances = np.clip(baseline_distances, 0.0, 2.0)

assert baseline_embeddings.shape[0] == len(reasoning) == 2534
print(f"Baseline population: {len(reasoning):,} reasoning paragraphs from {reasoning['document_id'].nunique()} documents")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Baseline population: 2,534 reasoning paragraphs from 99 documents


In [3]:
def candidate_summary(name: str, labels: np.ndarray) -> dict:
    labels = np.asarray(labels)
    assigned = labels >= 0
    assigned_labels = labels[assigned]
    _, counts = np.unique(assigned_labels, return_counts=True)
    cluster_count = len(counts)
    coverage = float(assigned.mean())
    if assigned.sum() > cluster_count >= 2:
        sil = float(silhouette_score(
            baseline_distances[np.ix_(assigned, assigned)],
            assigned_labels,
            metric="precomputed",
        ))
    else:
        sil = np.nan
    largest = int(counts.max()) if len(counts) else 0
    return {
        "configuration": name,
        "clusters": int(cluster_count),
        "assigned_nodes": int(assigned.sum()),
        "coverage": coverage,
        "largest_cluster": largest,
        "largest_cluster_share": largest / assigned.sum() if assigned.sum() else np.nan,
        "silhouette_assigned": sil,
        "coverage_adjusted_silhouette": coverage * sil if np.isfinite(sil) else np.nan,
    }


candidate_rows = []
candidate_labels = {}
for eps in (0.02, 0.04, 0.06, 0.08, 0.10, 0.12, 0.15):
    for min_samples in (3, 5, 10):
        name = f"DBSCAN eps={eps:.2f}, min_samples={min_samples}"
        labels = DBSCAN(eps=eps, min_samples=min_samples, metric="precomputed").fit_predict(baseline_distances)
        candidate_labels[name] = labels
        candidate_rows.append(candidate_summary(name, labels))

for min_cluster_size in (5, 10, 20):
    for min_samples in (3, 5, 10):
        name = f"HDBSCAN min_cluster_size={min_cluster_size}, min_samples={min_samples}"
        labels = HDBSCAN(
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            metric="precomputed",
            copy=True,
        ).fit_predict(baseline_distances.astype(np.float64))
        candidate_labels[name] = labels
        candidate_rows.append(candidate_summary(name, labels))

candidate_results = pd.DataFrame(candidate_rows).sort_values(
    ["coverage_adjusted_silhouette", "coverage", "silhouette_assigned"],
    ascending=False,
).reset_index(drop=True)
original_name = "DBSCAN eps=0.02, min_samples=10"
eligible = candidate_results.loc[
    candidate_results["clusters"].ge(2)
    & candidate_results["coverage"].ge(0.25)
    & candidate_results["largest_cluster_share"].le(0.15)
]
selected_name = eligible.iloc[0]["configuration"]
original_labels = candidate_labels[original_name]
selected_labels = candidate_labels[selected_name]

display(candidate_results.head(15).style.format({
    "coverage": "{:.1%}",
    "largest_cluster_share": "{:.1%}",
    "silhouette_assigned": "{:.3f}",
    "coverage_adjusted_silhouette": "{:.3f}",
}))
print(f"Selected density baseline: {selected_name}")

,configuration,clusters,assigned_nodes,coverage,largest_cluster,largest_cluster_share,silhouette_assigned,coverage_adjusted_silhouette
0,"HDBSCAN min_cluster_size=20, min_samples=10",3,2481,97.9%,2381,96.0%,0.553,0.541
1,"DBSCAN eps=0.15, min_samples=10",3,2477,97.8%,2382,96.2%,0.553,0.541
2,"DBSCAN eps=0.15, min_samples=5",3,2481,97.9%,2386,96.2%,0.552,0.540
3,"HDBSCAN min_cluster_size=20, min_samples=3",3,2505,98.9%,2405,96.0%,0.545,0.538
4,"DBSCAN eps=0.12, min_samples=10",3,2425,95.7%,2330,96.1%,0.562,0.538
5,"DBSCAN eps=0.12, min_samples=5",3,2428,95.8%,2333,96.1%,0.562,0.538
6,"HDBSCAN min_cluster_size=20, min_samples=5",3,2500,98.7%,2403,96.1%,0.545,0.538
7,"DBSCAN eps=0.15, min_samples=3",5,2488,98.2%,2386,95.9%,0.505,0.496
8,"DBSCAN eps=0.12, min_samples=3",6,2442,96.4%,2336,95.7%,0.372,0.359
9,"DBSCAN eps=0.08, min_samples=10",5,2089,82.4%,1943,93.0%,0.368,0.303


Selected density baseline: HDBSCAN min_cluster_size=5, min_samples=3


## Common evaluation

Noise is excluded from cluster-quality calculations and retained in the reported coverage denominator. Cohesion is the pair-weighted mean cosine similarity inside non-singleton clusters. The silhouette is computed only over assigned observations. Document uniqueness measures whether a cluster contains at most one item per source judgment; the graph-aware notebook enforces this property by construction.

In [4]:
def evaluate_partition(
    name: str,
    embeddings: np.ndarray,
    labels: np.ndarray,
    document_ids: np.ndarray,
    type_labels: np.ndarray | None = None,
) -> dict:
    labels = np.asarray(labels)
    document_ids = np.asarray(document_ids).astype(str)
    assigned = labels >= 0
    assigned_indices = np.flatnonzero(assigned)
    assigned_labels = labels[assigned]
    unique_labels, counts = np.unique(assigned_labels, return_counts=True)
    cluster_count = len(unique_labels)

    pair_similarity_sum = 0.0
    pair_count = 0
    document_unique_clusters = 0
    type_majority = 0
    assigned_frame = pd.DataFrame({
        "index": assigned_indices,
        "label": assigned_labels,
        "document_id": document_ids[assigned],
    })
    if type_labels is not None:
        assigned_frame["type"] = np.asarray(type_labels)[assigned]

    for _, group in assigned_frame.groupby("label", sort=False):
        indices = group["index"].to_numpy()
        size = len(indices)
        document_unique_clusters += int(group["document_id"].nunique() == size)
        if type_labels is not None:
            type_majority += int(group["type"].value_counts().iloc[0])
        if size > 1:
            similarities = embeddings[indices] @ embeddings[indices].T
            pair_similarity_sum += float(np.triu(similarities, k=1).sum())
            pair_count += size * (size - 1) // 2

    sil = np.nan
    if assigned.sum() > cluster_count >= 2:
        sil = float(silhouette_score(
            embeddings[assigned],
            assigned_labels,
            metric="cosine",
        ))

    return {
        "method": name,
        "population": len(labels),
        "clusters": cluster_count,
        "assigned_nodes": int(assigned.sum()),
        "coverage": float(assigned.mean()),
        "singleton_nodes": int(counts[counts == 1].sum()),
        "singleton_node_rate": float(counts[counts == 1].sum() / len(labels)),
        "mean_cluster_size": float(counts.mean()),
        "largest_cluster": int(counts.max()),
        "silhouette_assigned": sil,
        "mean_within_cosine": pair_similarity_sum / pair_count if pair_count else np.nan,
        "document_unique_clusters": document_unique_clusters / cluster_count,
        "type_purity": type_majority / assigned.sum() if type_labels is not None else np.nan,
    }


later = pd.read_parquet(ARTIFACT_DIR / "later_clusters.parquet").sort_values("global_node_index")
later_embeddings = np.load(ARTIFACT_DIR / "later_text_embeddings.npy")
later_diagnostics = pd.read_parquet(ARTIFACT_DIR / "later_partition_diagnostics.parquet")
later_affinity = sparse.load_npz(ARTIFACT_DIR / "later_affinity_graph.npz")

assert np.array_equal(later["global_node_index"].to_numpy(), np.arange(len(later)))
assert later_embeddings.shape[0] == len(later) == 3941

comparison = pd.DataFrame([
    evaluate_partition(
        "1.0 original DBSCAN", baseline_embeddings, original_labels,
        reasoning["document_id"].to_numpy(),
    ),
    evaluate_partition(
        "1.0 tuned density baseline", baseline_embeddings, selected_labels,
        reasoning["document_id"].to_numpy(),
    ),
    evaluate_partition(
        "1.7 graph-aware partition", later_embeddings,
        later["cluster_id"].to_numpy(), later["document_id"].to_numpy(),
        later["argument_type"].to_numpy(),
    ),
]).set_index("method")

display(comparison.style.format({
    "coverage": "{:.1%}",
    "singleton_node_rate": "{:.1%}",
    "silhouette_assigned": "{:.3f}",
    "mean_within_cosine": "{:.3f}",
    "document_unique_clusters": "{:.1%}",
    "type_purity": "{:.1%}",
}))

,population,clusters,assigned_nodes,coverage,singleton_nodes,singleton_node_rate,mean_cluster_size,largest_cluster,silhouette_assigned,mean_within_cosine,document_unique_clusters,type_purity
method,,,,,,,,,,,,
1.0 original DBSCAN,2534,12,201,7.9%,0,0.0%,16.750000,47,0.939,0.997,58.3%,nan%
1.0 tuned density baseline,2534,77,1119,44.2%,0,0.0%,14.532468,96,0.497,0.954,46.8%,nan%
1.7 graph-aware partition,3941,1121,3941,100.0%,591,15.0%,3.515611,45,-0.244,0.905,100.0%,100.0%


In [5]:
later_cluster_sizes = later.groupby("cluster_id").size()
later_multi = later.loc[later["cluster_id"].isin(later_cluster_sizes[later_cluster_sizes > 1].index)]

quality_checks = pd.Series({
    "later_nodes_covered_once": later["global_node_index"].nunique() == len(later) == 3941,
    "later_type_pure": comparison.loc["1.7 graph-aware partition", "type_purity"] == 1.0,
    "later_document_unique": comparison.loc["1.7 graph-aware partition", "document_unique_clusters"] == 1.0,
    "later_sparse_graph_symmetric": (later_affinity - later_affinity.T).nnz == 0,
    "later_multi_node_clusters": int((later_cluster_sizes > 1).sum()),
    "later_singleton_clusters": int((later_cluster_sizes == 1).sum()),
    "later_nodes_in_multi_node_clusters": int((later_cluster_sizes[later_cluster_sizes > 1]).sum()),
    "later_median_multi_cluster_conductance": float(
        later_diagnostics.loc[later_diagnostics["size"] > 1, "conductance"].median()
    ),
}, name="value")
display(quality_checks.to_frame())

,value
later_nodes_covered_once,True
later_type_pure,True
later_document_unique,True
later_sparse_graph_symmetric,True
later_multi_node_clusters,530
later_singleton_clusters,591
later_nodes_in_multi_node_clusters,3350
later_median_multi_cluster_conductance,0.426183


In [6]:
representative_ids = (
    later_diagnostics.loc[later_diagnostics["size"].between(5, 20)]
    .sort_values(["conductance", "size"], ascending=[True, False])
    .head(3)["cluster_id"]
)
qualitative_sample = later.loc[
    later["cluster_id"].isin(representative_ids),
    ["cluster_id", "cluster_size", "document_id", "argument_type", "argumentative_role", "text"],
].copy()
qualitative_sample["text"] = qualitative_sample["text"].str.replace(r"\s+", " ", regex=True).str.slice(0, 220)
display(qualitative_sample.head(30))

,cluster_id,cluster_size,document_id,argument_type,argumentative_role,text
3182,575,5,116062204,CLAIM,FINAL_CONCLUSION,Долю речових доказів суд вирішує у відповіднос...
425,25,18,116102314,CLAIM,FINAL_CONCLUSION,Питання про речові докази по справі вирішити у...
386,22,14,116109598,PREMISE,BASE_PREMISE,"Статтею 50 КК України передбачено, що покаранн..."
3181,575,5,116110555,CLAIM,FINAL_CONCLUSION,Питання про речові докази суд вирішує відповід...
387,22,14,116116081,PREMISE,BASE_PREMISE,Згідно із ч.2 ст.50 КК України покарання має н...
426,25,18,116116081,CLAIM,FINAL_CONCLUSION,Питання щодо речових доказів суд вирішує відпо...
388,22,14,116120654,PREMISE,BASE_PREMISE,Відповідно до ч. 2 ст. 50 КК України покарання...
427,25,18,116126076,CLAIM,FINAL_CONCLUSION,Долю речових доказів вирішити відповідно до ст...
389,22,14,116170822,PREMISE,BASE_PREMISE,При призначенні покарання суд враховує положен...
390,22,14,116192026,PREMISE,BASE_PREMISE,"покарання має на меті не тільки кару, а й випр..."


## Interpretation

**For the intended cross-document argument-alignment task, 1.7 is a clear structural improvement, but it is not better on every embedding-only metric.** It covers all 3,941 argument nodes exactly once; 3,350 nodes belong to 530 multi-node clusters and 591 nodes remain explicit singletons. Every cluster is argument-type pure and contains at most one node from each source document. Median conductance among multi-node clusters is 0.426.

The saved 1.0 DBSCAN configuration assigns only 201 of 2,534 paragraphs (7.9%). Its assigned subset is extremely tight (mean within-cluster cosine 0.997; silhouette 0.939), but treating noise as another label gives the misleading saved silhouette of -0.316. The tuned HDBSCAN baseline improves useful paragraph coverage to 1,119 paragraphs (44.2%) across 77 clusters while keeping the largest cluster to 8.6% of assigned items; its assigned silhouette is 0.497 and mean within-cluster cosine is 0.954.

The 1.7 partition has lower embedding-only separation (silhouette -0.244) and mean within-cluster cosine 0.905. This occurs because it makes a full, fine-grained partition using graph structure, section information, argument type, and relation propagation; nearby semantic regions can be divided into several valid graph-aware clusters, while singletons receive silhouette zero. Consequently, the later result is improved for complete and valid argument alignment, while the tuned 1.0 result remains stronger if the goal is only high-precision semantic paragraph grouping with unassigned noise allowed.

This is not a controlled same-population benchmark: 1.0 clusters reasoning paragraphs and 1.7 clusters extracted argument components. The conclusion therefore rests on the intended downstream behavior and the reported trade-offs rather than on a single score.